In [1]:
%load_ext autoreload
%autoreload 2

import os
import glob
import yaml
import warnings; warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import sys
sys.path.append('..') # src 폴더 인식을 위한 경로 추가
from src import build_parent_retriever, run_corrective_agent

In [2]:
load_dotenv(dotenv_path="../.env") 
with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

gemini_api_key = os.getenv(config['api_keys']['google_gemini_env_name'])
if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("✅ Gemini API 키 로드 완료")
else:
    print("❌ API 키 오류")

✅ Gemini API 키 로드 완료


In [3]:
data_dir = "../data/raw/"
pdf_files = glob.glob(os.path.join(data_dir, "*.pdf"))

print(f"📂 {len(pdf_files)}개의 PDF 문서에 대한 고급 검색 엔진을 구축합니다...")

# src/document_processor.py 에 정의된 함수 단 한 줄로 복잡한 DB 구축 끝!
retriever, all_docs = build_parent_retriever(pdf_files, chunk_size=400)

📂 10개의 PDF 문서에 대한 고급 검색 엔진을 구축합니다...


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\document_processor.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39816.18it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ 하드디스크에 저장된 DB와 문서를 1초 만에 불러옵니다!


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\document_processor.py:27: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [4]:
# ==========================================
# 🧠 동적 에이전트 루프 실행 (다중 후보 순회 탐색 적용)
# ==========================================
print("\n" + "="*50)
print("🧐 다중 문서 환경에서의 하이브리드 에이전트 테스트")

query = "재무제표 기준 삼성SDI의 25년도 당기순손익은 얼마인가요?"
print(f"질문: {query}\n")

# --- Step 1: 텍스트 기반 후보군 탐색 ---
print("🔍 [Path A] 10개 기업의 수만 개 조각 중 목표 위치 후보들을 탐색합니다...")
retriever.search_kwargs = {"k": 20}
retrieved_docs = retriever.invoke(query)

if not retrieved_docs:
    print("❌ 관련된 문서를 찾지 못했습니다.")
else:
    print(f"🎯 총 {len(retrieved_docs)}개의 유력한 후보 페이지 묶음을 발견했습니다!\n")
    final_answer = ""
    
    for i, doc in enumerate(retrieved_docs):
        target_pdf = doc.metadata.get('source')
        start_page = doc.metadata.get('page', 0)
        
        print("-" * 50)
        print(f"▶️ [후보 {i+1}/{len(retrieved_docs)}] {os.path.basename(target_pdf)}의 {start_page + 1}p부터 탐색")
        
        answer = run_corrective_agent(query, target_pdf, start_page, all_docs, max_expansions=5)
        
        if answer == "WRONG_DOCUMENT":
            print("⏭️ 결과: 기업명이 다름. 다음 후보로 점프!")
            continue
        elif answer == "NOT_FOUND_IN_THIS_CANDIDATE":
            print("⏭️ 결과: 해당 위치에 정답 없음. 다음 후보로 점프!")
            continue
        else:
            print("🎉 결과: 정답 발견!")
            final_answer = answer
            break

    if not final_answer:
        final_answer = "모든 문서와 후보 페이지를 정밀 조사했으나 당기순이익 수치를 찾지 못했습니다."

    print("\n🏆 최종 답변:\n", final_answer)



🧐 다중 문서 환경에서의 하이브리드 에이전트 테스트
질문: 재무제표 기준 삼성SDI의 25년도 당기순손익은 얼마인가요?

🔍 [Path A] 10개 기업의 수만 개 조각 중 목표 위치 후보들을 탐색합니다...
🎯 총 18개의 유력한 후보 페이지 묶음을 발견했습니다!

--------------------------------------------------
▶️ [후보 1/18] samsung_sdi_report_2026.pdf의 46p부터 탐색

🔄 [Agent Loop 1] 분석 중인 페이지: [46, 47, 48]
🤔 Gemini 3.1 Flash Lite가 검증 중...
🚨 결과: 표 이어짐 -> 49p로 확장

🔄 [Agent Loop 2] 분석 중인 페이지: [46, 47, 48, 49]
🤔 Gemini 3.1 Flash Lite가 검증 중...
🚨 결과: 표 이어짐 -> 50p로 확장

🔄 [Agent Loop 3] 분석 중인 페이지: [46, 47, 48, 49, 50]
🤔 Gemini 3.1 Flash Lite가 검증 중...
🚨 결과: 표 이어짐 -> 51p로 확장

🔄 [Agent Loop 4] 분석 중인 페이지: [46, 47, 48, 49, 50, 51]
🤔 Gemini 3.1 Flash Lite가 검증 중...
🚨 결과: 표 이어짐 -> 52p로 확장

🔄 [Agent Loop 5] 분석 중인 페이지: [46, 47, 48, 49, 50, 51, 52]
🤔 Gemini 3.1 Flash Lite가 검증 중...
🚨 결과: 표 이어짐 -> 53p로 확장

🔄 [Agent Loop 6] 분석 중인 페이지: [46, 47, 48, 49, 50, 51, 52, 53]
🤔 Gemini 3.1 Flash Lite가 검증 중...
⚠️ 확장 한도 초과
⏭️ 결과: 해당 위치에 정답 없음. 다음 후보로 점프!
--------------------------------------------------
▶️ [후보 2/18] samsung_electr